# SQLAlchemy: Database ORM

**Purpose:** Object-relational mapping for database operations.

← [Modules](./README.md) | **01. SQLAlchemy** | [02. Pydantic →](./02-pydantic.ipynb)

## Simple: Basic CRUD

In [1]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, Session

Base = declarative_base()

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    name = Column(String)
    email = Column(String)

engine = create_engine("sqlite:///app.db")
Base.metadata.create_all(engine)

# Create
session = Session(engine)
user = User(name="Alice", email="alice@example.com")
session.add(user)
session.commit()

print(f"Created user: {user.id}")

Created user: 3


In [2]:
# Read
user = session.query(User).filter(User.name == "Alice").first()
print(f"Found user: {user.name} ({user.email})")

Found user: Alice (alice@example.com)


In [3]:
# Update
user.email = "alice.new@example.com"
session.commit()
print(f"Updated email: {user.email}")

Updated email: alice.new@example.com


In [ ]:
# Delete
session.delete(user)
session.commit()
print("User deleted")

## Medium: Relationships & Filtering

In [4]:
from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

class Post(Base):
    __tablename__ = "posts"
    id = Column(Integer, primary_key=True)
    title = Column(String)
    user_id = Column(Integer, ForeignKey("users.id"))

User.posts = relationship("Post")
Base.metadata.create_all(engine)

# Create users and posts
user1 = User(name="Bob", email="bob@example.com")
session.add(user1)
session.commit()

post1 = Post(title="First Post", user_id=user1.id)
post2 = Post(title="Second Post", user_id=user1.id)
session.add_all([post1, post2])
session.commit()

print(f"Created {len(user1.posts)} posts for {user1.name}")

Created 2 posts for Bob


In [5]:
# Query with relationship
user = session.query(User).filter(User.name == "Bob").first()
for post in user.posts:
    print(f"  - {post.title}")

  - Updated Title
  - Second Post


In [ ]:
# Bulk operations
session.query(Post).filter(Post.title.like("%First%")).update({"title": "Updated Title"})
session.commit()

user = session.query(User).filter(User.name == "Bob").first()
for post in user.posts:
    print(f"  - {post.title}")

## Complex: Transactions & Events

In [ ]:
from sqlalchemy import event
from sqlalchemy.exc import IntegrityError

class Account(Base):
    __tablename__ = "accounts"
    id = Column(Integer, primary_key=True)
    balance = Column(Integer, default=0)

Base.metadata.create_all(engine)

@event.listens_for(Session, "after_insert")
def log_insert(mapper, connection, target):
    print(f"Inserted: {target.__class__.__name__}")

# Test transaction
acc1 = Account(balance=1000)
acc2 = Account(balance=500)
session.add_all([acc1, acc2])
session.commit()
print(f"Accounts created: {acc1.id}, {acc2.id}")

In [ ]:
# Transaction with error handling
def transfer(from_id, to_id, amount):
    try:
        from_acc = session.query(Account).with_for_update().filter(Account.id == from_id).first()
        to_acc = session.query(Account).with_for_update().filter(Account.id == to_id).first()
        
        from_acc.balance -= amount
        to_acc.balance += amount
        session.commit()
        print(f"Transferred {amount} from account {from_id} to {to_id}")
    except IntegrityError as e:
        session.rollback()
        print(f"Transfer failed: {e}")

transfer(acc1.id, acc2.id, 100)

# Verify
acc1_updated = session.query(Account).filter(Account.id == acc1.id).first()
acc2_updated = session.query(Account).filter(Account.id == acc2.id).first()
print(f"Account 1 balance: {acc1_updated.balance}")
print(f"Account 2 balance: {acc2_updated.balance}")

---

**Install:** `pip install sqlalchemy` | **Use:** Web apps, APIs, data pipelines

← [Modules](./README.md) | **01. SQLAlchemy** | [02. Pydantic →](./02-pydantic.ipynb)

## Resources

- [SQLAlchemy Documentation](https://docs.sqlalchemy.org/)
- [SQLAlchemy ORM Tutorial](https://docs.sqlalchemy.org/en/20/orm/)
